# Engine: H = xp — the lossless semantic engine

**File:** `ValaQuenta/hamiltonian.py`
**Wiki:** [wiki/hamiltonian.md](../../wiki/hamiltonian.md)

Berry & Keating, 1999. The classical orbit `xp = E` is a hyperbola, and its
equations of motion have no loss term: `x_dot = x`, `p_dot = -p`. The energy
`E = xp` is conserved exactly, which is what makes it a *lossless* engine.

Three Hamiltonians live in this file: `HamiltonianXP`, the lemniscatic
`FermatEllipticHamiltonian`, and `RedBlueHamiltonian`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../../..'))
import math, cmath
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10,
                     'axes.spines.top': False, 'axes.spines.right': False})
print('python', sys.version.split()[0])

In [ ]:
from ValaQuenta import (HamiltonianXP, FermatEllipticHamiltonian,
                        RedBlueHamiltonian, RIEMANN_ZEROS)

H = HamiltonianXP()

# Scale invariance: (x,p) -> (lam*x, p/lam) leaves xp unchanged.
print('scale_check(2, 3, lam=2) ->', H.scale_check(2, 3, lam=2.0))

# One unit of time from (1,1): x should be e, p should be 1/e, xp still 1.
x, p = H.trajectory(1.0, 1.0, t=1.0)
print(f'trajectory(1,1,t=1) -> x={x!r}  p={p!r}')
print(f'  x  vs e    : {x!r} vs {math.e!r}   diff={abs(x-math.e):.3e}')
print(f'  p  vs 1/e  : {p!r} vs {1/math.e!r}  diff={abs(p-1/math.e):.3e}')
print(f'  E = x*p    : {x*p!r}   (conserved: started at 1.0)')

`x` lands on **e** and `p` on **1/e** after one unit of time. That is not
a coincidence to be admired — it is what `x_dot = x` means. It is shown here
because it is the cheapest possible check that the integrator is exact.

In [ ]:
# Conservation across a long trajectory.
ts = np.linspace(0, 4, 60)
xs, ps, Es = [], [], []
for t in ts:
    xi, pi = H.trajectory(1.0, 1.0, t=float(t))
    xs.append(xi); ps.append(pi); Es.append(xi * pi)
Es = np.array(Es)
print(f'E over t in [0,4]:  min={Es.min()!r}  max={Es.max()!r}')
print(f'max drift from 1.0: {np.abs(Es-1.0).max():.3e}')

fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.6))
a1.plot(xs, ps, color='#3f7fb0')
a1.set_xlabel('x'); a1.set_ylabel('p'); a1.set_title('xp = E  (the hyperbola)')
a1.set_xscale('log'); a1.set_yscale('log')
a2.plot(ts, Es - 1.0, color='#b04a3f')
a2.set_xlabel('t'); a2.set_ylabel('E - 1'); a2.set_title('conservation drift')
a2.ticklabel_format(axis='y', style='sci', scilimits=(0, 0))
plt.tight_layout(); plt.show()

## The Riemann zeros

In [ ]:
z = H.zeros(n=5)
print('first 5 BK zeros:')
for i, g in enumerate(z, 1):
    print(f'  gamma_{i} = {g!r}')
print()
print('RIEMANN_ZEROS (module table), first 5:')
print(' ', RIEMANN_ZEROS[:5])

## Fermat elliptic (lemniscatic case)

`g2 = 1, g3 = 0` is the lemniscatic case. The discriminant
`Delta = g2^3 - 27*g3^2` must be non-zero for a valid elliptic curve.

In [ ]:
F = FermatEllipticHamiltonian(g2=1.0, g3=0.0)
print(f'discriminant Delta = {F.discriminant()!r}   (non-zero -> valid curve)')
print(f'weierstrass_p(1.0) = {F.weierstrass_p(1.0)!r}')

## RedBlue — balance at sigma = 1/2

The Red (Riemann / ascending) and Blue (Fermat / descending) currents are equal
where the geometry is balanced. That point is sigma = 1/2.

In [ ]:
RB = RedBlueHamiltonian()
fwd = RB.noether_forward(0.5, 0.5)
bwd = RB.noether_backward(0.5, 0.5)
print(f'noether_forward (0.5,0.5) = {fwd!r}')
print(f'noether_backward(0.5,0.5) = {bwd!r}')
print(f'balance         (0.5,0.5) = {RB.balance(0.5,0.5)!r}')
print(f'functional_equation_check = {RB.functional_equation_check(0.5,0.5)!r}')